# Wyjątki i klasy abstrakcyjne

### Hierarchia wyjątków

Mechanizm wyłapywania wyjątków bazuje na dziedziczeniu. Np. hierarchia standardowych wyjątków ([Exception hierarchy &mdash; Built-in Exceptions &mdash; Python documentation](https://docs.python.org/3/library/exceptions.html#exception-hierarchy)) jest właśnie hierarchią klas &mdash; nazwa wyjątku to nazwa klasy. Klauzula `except` wyłapuje wyjątki wymienionych w niej klas i ich klas pochodnych.

In [ ]:
try:
    raise IndexError()
except LookupError:
    print("Błąd!")

Definiowanie swoich własnych typów wyjątków polega po prostu na definiowanu odpowiednich klas. Takie klasy powinny dziedziczyć bezpośrednio lub pośrednio po klasie [`Exception`](https://docs.python.org/3/library/exceptions.html#Exception). Często cała definicja klasy wyjątku będzie wyglądała tak:

In [ ]:
class BladUjemnejWartosci(ValueError):
    pass

Najważniejsza informacja związana z wyjątkiem, to zazwyczaj po prostu typ wyjątku.

In [ ]:
def predkosc(droga, czas):
    if droga < 0:
        raise BladUjemnejWartosci("ujemna droga")
    if czas < 0:
        raise BladUjemnejWartosci("ujemny czas")
    if czas == 0:
        raise ValueError("czas musi być dodatni")
    return droga / czas

Możemy jej teraz użyć tak:

In [ ]:
try:
    droga = float(input("droga: "))
    czas = float(input("czas: "))
    print(predkosc(droga, czas))
except ValueError as e:
    print("Błędne wartości!")

Widzimy, że wpisanie ujemnych wartości powoduje wyjątek klasy `BladUjemnejWartosci`. Jest on wyłapywany, ponieważ jest to podklasa klasy [`ValueError`](https://docs.python.org/3/library/exceptions.html#ValueError).

Błąd [`ValueError`](https://docs.python.org/3/library/exceptions.html#ValueError) może być tutaj spowodowany również wpisaniem przez użytkownika napisu, którego nie da się zinterpretować jako wartości typu [`float`](https://docs.python.org/3/library/functions.html#float) (wypróbuj to). Powyższy sposób obsługi wyjątków nie rozróżnia tych sytuacji. Poniższy sposób rozróżnia:

In [ ]:
try:
    droga = float(input("droga: "))
    czas = float(input("czas: "))
    print(predkosc(droga, czas))
except BladUjemnejWartosci as e:
    print(f"Błędna wartość: {e}")
except ValueError:
    print("Wpisane wartości nie są liczbami.")

Odwrócenie kolejności klauzul `except` w powyższym przykładzie jest błędem. Efektem byłoby to, że druga z nich nie zostałaby nigdy użyta &mdash; wyjątek `BladUjemnejWartosci` zostałby wyłapany przez pierwszą z nich.

In [ ]:
try:
    droga = float(input("droga: "))
    czas = float(input("czas: "))
    print(predkosc(droga, czas))
except ValueError:
    print("Wpisane wartości nie są liczbami.")
except BladUjemnejWartosci as e:
    print(f"Błędna wartość: {e}")

### Klasy abstrakcyjne

Wróćmy do przykładu z klasami `Kolo` i `Kwadrat` oraz funkcją `wyswietl_figure`. Chcemy stworzyć hierarchię klas dla figur podobną do hierarchii dla osób:

```text
Figura
├── Kolo
└── Kwadrat
```

Wtedy w funkcji `wyswietl_figure` będzie można sprawdzić za pomocą funkcji [`isinstance`](https://docs.python.org/3/library/functions.html#isinstance), czy argument jest figurą. Zdefiniujmy odpowiednie trzy klasy.

Metod `obwod` i `pole` nie da się sensownie zaimplementować w klasie `Figura`. Moglibyśmy w związku z tym pominąć ich implementację
lub wyrzucić w nich odpowiedni wyjątek ([`NotImplementedError`](https://docs.python.org/3/library/exceptions.html#NotImplementedError)) i zaimplementować je w podklasach.

In [ ]:
import math


class Figura:
    def obwod(self):
        raise NotImplementedError("brak implementacji metody obwod")

    def pole(self):
        raise NotImplementedError("brak implementacji metody pole")


class Kolo(Figura):
    def __init__(self, promien):
        self.promien = promien

    def obwod(self):
        return 2 * math.pi * self.promien

    def pole(self):
        return math.pi * self.promien**2


class Kwadrat(Figura):
    def __init__(self, bok):
        self.bok = bok

    def obwod(self):
        return 4 * self.bok

    def pole(self):
        return self.bok**2


def wyswietl_figure(figura):
    if not isinstance(figura, Figura):
        raise TypeError("potrzebny obiekt klasy Figura")
    print(f"obwód: {figura.obwod()}")
    print(f"pole: {figura.pole()}")

In [ ]:
kwadrat = Kwadrat(5)
kolo = Kolo(3)

Obiekty `kwadrat` i `kolo` są egzemplarzami klas pochodnych od `Figura`, więc spełniają również sprawdzenie `isinstance(..., Figura)`.

In [ ]:
wyswietl_figure(kwadrat)

In [ ]:
wyswietl_figure(kolo)

Zdefiniujmy klasę z metodami `pole` i `obwod`, ale niebędącą podklasą klasy `Figura`.

In [ ]:
class A:
    def pole(self):
        return 3

    def obwod(self):
        return 5

In [ ]:
a = A()

In [ ]:
a.obwod()

In [ ]:
a.pole()

Obiekt `a` ma wprawdzie metody `pole` i `obwod`, ale jego klasa `A` nie dziedziczy po `Figura`. Dlatego sprawdzenie w funkcji `wyswietl_figure` odrzuci go, zgłaszając [`TypeError`](https://docs.python.org/3/library/exceptions.html#TypeError):

In [ ]:
wyswietl_figure(a)

Problemem jest teraz to, że możemy utworzyć obiekty typu `Figura`, co nie ma sensu, bo nie możemy nic z takim obiektem zrobić &mdash; nie ma sensu liczenie pola lub obwodu jeżeli nie znamy konkretnego typu figury.

In [ ]:
figura = Figura()

Obiekt `figura` można jeszcze utworzyć, ale próba obliczenia jego obwodu zakończy się [`NotImplementedError`](https://docs.python.org/3/library/exceptions.html#NotImplementedError). Metoda klasy `Figura` nie zawiera obliczenia, tylko zgłasza ten wyjątek:

In [ ]:
wyswietl_figure(figura)

Klasa `Figura` jest nam potrzebna tylko po to, żeby móc utworzyć hierarchię klas, ale każda figura powinna mieć jakiś **konkretny** typ &mdash; `Kwadrat` lub `Kolo`. Nie powinno być obiektów typu `Figura`. Żeby policzyć pole lub obwód musimy znać ten konkretny typ &mdash; każdy klasa implementująca konkretną figurę dostarcza odpowiednią implementację.

Metody takie jak `pole` i `obwod` nazywa się metodami **abstrakcyjnymi** &mdash; każda figura ma pole i obwód (każda instancja klasy `Figura` będzie miała odpowiednie metody), ale nie wiemy jak je policzyć dopóki nie znamy konkretnego typu figury.

Klasę taką jak `Figura` nazywa się **klasą abstrakcyjną**. Klasy takie jak `Kolo` i `Kwadrat`, to tzw. **klasy konkretne**.

Jeżeli chcemy wprost wyrazić to, że klasa jest abstrakcyjna, to możemy użyć modułu [`abc`](https://docs.python.org/3/library/abc.html) (skrót od ang. *abstract base classes* &mdash; abstrakcyjne klasy bazowe). Klasa abstrakcyjna powinna dziedziczyć po klasie [`abc.ABC`](https://docs.python.org/3/library/abc.html#abc.ABC), a metody abstrakcyjne powinny być oznaczone dekoratorem [`abc.abstractmethod`](https://docs.python.org/3/library/abc.html#abc.abstractmethod).

In [ ]:
import abc
import math


class Figura(abc.ABC):
    @abc.abstractmethod
    def obwod(self):
        pass

    @abc.abstractmethod
    def pole(self):
        pass


class Kolo(Figura):
    def __init__(self, promien):
        self.promien = promien

    def obwod(self):
        return 2 * math.pi * self.promien

    def pole(self):
        return math.pi * self.promien**2


class Kwadrat(Figura):
    def __init__(self, bok):
        self.bok = bok

    def obwod(self):
        return 4 * self.bok

    def pole(self):
        return self.bok**2


def wyswietl_figure(figura):
    if not isinstance(figura, Figura):
        raise TypeError("potrzebny obiekt klasy Figura")
    print(f"obwód: {figura.obwod()}")
    print(f"pole: {figura.pole()}")

In [ ]:
kwadrat = Kwadrat(5)

In [ ]:
wyswietl_figure(kwadrat)

Teraz błąd jest zgłaszany już przy próbie utworzenia obiektu klasy abstrakcyjnej, a nie dopiero przy próbie wywołania metody.

Nie można już utworzyć obiektu samej klasy `Figura`: jej metody `pole` i `obwod` są abstrakcyjne. Poniższa próba zakończy się wyjątkiem [`TypeError`](https://docs.python.org/3/library/exceptions.html#TypeError):

In [ ]:
f = Figura()

Jeżeli podklasa klasy abstrakcyjnej nie dostarczy implementacji wszystkich metod abstrakcyjnych, to sama będzie klasą abstrakcyjną.

In [ ]:
class Rownoleglobok(Figura):
    def __init__(self, a, b):
        self.a = a
        self.b = b

    def obwod(self):
        return 2 * (self.a + self.b)

Klasa `Rownoleglobok` definiuje `obwod`, ale nie implementuje abstrakcyjnej metody `pole`. Nadal jest więc klasą abstrakcyjną i próba utworzenia jej obiektu spowoduje [`TypeError`](https://docs.python.org/3/library/exceptions.html#TypeError):

In [ ]:
rownoleglobok = Rownoleglobok(3, 4)

In [ ]:
class Prostokat(Rownoleglobok):
    def __init__(self, a, b):
        super().__init__(a, b)

    def pole(self):
        return self.a * self.b

In [ ]:
prostokat = Prostokat(3, 4)

In [ ]:
prostokat.obwod()

In [ ]:
prostokat.pole()